# 📄 DocMatcher: Document Image Dewarping via Structural and Textual Line Matching

## Result Reproduction Notebook (Kaggle Version)

**Paper:** [DocMatcher](https://felixhertlein.github.io/doc-matcher/) — IEEE/CVF WACV 2025

**Authors:** Felix Hertlein, Alexander Naumann, York Sure-Vetter

**Repository:** [GitHub - FelixHertlein/doc-matcher](https://github.com/FelixHertlein/doc-matcher)

---

### Overview

DocMatcher is a novel approach for document image dewarping that leverages **structural and textual line matching** between warped document images and their flat templates. The pipeline consists of:

1. **Segmentation** (SAM) — Segment the document region
2. **Line Detection** (LineFormer) — Detect text/structural lines
3. **Pre-unwarp Homography** — Initial perspective correction
4. **Line Matching** (LightGlue) — Match lines between warped image and template
5. **Unwarp via Correspondence** — Final dewarping using matched correspondences

This notebook reproduces inference results on the **Inv3DReal** dataset and compares DocMatcher against baseline methods.

> ⚠️ **Before running:** Go to **Settings (right panel) → Accelerator → GPU T4 x1**, then **Save** and re-run.

---
## 1. Environment Setup

This project requires **Python 3.10** (for PyTorch 1.13.1 + mmcv-full 1.7.2 compatibility).

Kaggle already runs Python 3.10, but we set up a **Python 3.10 virtual environment** inside `/kaggle/working/` to isolate all heavy dependencies cleanly and avoid conflicts with Kaggle's pre-installed packages.

### 1.1 Install Python 3.10 & Create Virtual Environment

In [ ]:
%%bash
set -e

echo "=== Wiping old venv ==="
rm -rf /kaggle/working/py310env

echo "=== Installing Python 3.10 ==="
if ! command -v python3.10 &> /dev/null; then
    apt-get update -qq
    apt-get install -y -qq software-properties-common > /dev/null 2>&1
    add-apt-repository -y ppa:deadsnakes/ppa > /dev/null 2>&1
    apt-get update -qq
    apt-get install -y -qq python3.10 python3.10-venv python3.10-dev python3.10-distutils > /dev/null 2>&1
else
    apt-get install -y -qq python3.10-venv python3.10-dev python3.10-distutils > /dev/null 2>&1 || true
fi

echo "=== Creating fresh venv ==="
python3.10 -m venv /kaggle/working/py310env

echo "=== Upgrading pip ==="
/kaggle/working/py310env/bin/pip install --upgrade pip -q

echo "=== Installing system deps ==="
apt-get install -y -qq tesseract-ocr libgl1-mesa-glx libglib2.0-0 > /dev/null 2>&1

echo ""
echo "✅ Fresh Python 3.10 venv ready"
/kaggle/working/py310env/bin/python --version

### 1.2 Clone Repository

In [ ]:
import os

if not os.path.exists('/kaggle/working/doc-matcher'):
    !git clone https://github.com/FelixHertlein/doc-matcher.git /kaggle/working/doc-matcher
    print("✅ Repository cloned")
else:
    print("✅ Repository already exists")

%cd /kaggle/working/doc-matcher

In [ ]:
# Fix GPU device mismatch in both inference files
!sed -i '/model.eval()/i\    model = model.cuda(gpu)' /kaggle/working/doc-matcher/src/unwarp_geotr/inference.py
!sed -i '/model.eval()/i\    model = model.to(device)' /kaggle/working/doc-matcher/src/line_matching/line_lightglue/inference.py

In [ ]:
!/kaggle/working/py310env/bin/pip install "huggingface_hub<0.24.0" -q

### 1.3 Install All Dependencies into Python 3.10 venv

Installing the exact dependency stack matching the paper's Docker environment.

In [ ]:
%%bash
set -e
PIP=/kaggle/working/py310env/bin/pip

echo "=== [1/7] Installing PyTorch 1.13.1 + CUDA 11.7 ==="
$PIP install torch==1.13.1+cu117 torchvision==0.14.1+cu117 --extra-index-url https://download.pytorch.org/whl/cu117 -q
echo "✅ PyTorch installed"

echo ""
echo "=== [2/7] Installing mmcv-full 1.7.2 ==="
$PIP install https://download.openmmlab.com/mmcv/dist/cu117/torch1.13.0/mmcv_full-1.7.2-cp310-cp310-manylinux1_x86_64.whl -q
echo "✅ mmcv-full installed"

echo ""
echo "=== [3/7] Installing mmdet 2.28.1 ==="
$PIP install mmdet==2.28.1 -q
echo "✅ mmdet installed"

echo ""
echo "=== [4/7] Installing python-doctr (WITHOUT torch upgrade) ==="
$PIP install "python-doctr[torch]==0.7.0" --no-deps -q
$PIP install mplcursors matplotlib scipy rapidfuzz defusedxml anyascii certifi webencodings tqdm -q
echo "✅ python-doctr installed"

echo ""
echo "=== [5/7] Installing Levenshtein ==="
$PIP install Levenshtein -q
echo "✅ Levenshtein installed"

echo ""
echo "=== [6/7] Installing remaining requirements ==="
cd /kaggle/working/doc-matcher
grep -v 'python_Levenshtein' requirements.txt \
  | grep -v 'python_doctr' \
  | grep -v -i 'torch' \
  | grep -v 'pytorch_lightning' \
  > /tmp/requirements_filtered.txt
$PIP install -r /tmp/requirements_filtered.txt -q
$PIP install "pytorch_lightning==1.9.5" --no-deps -q
echo "✅ Remaining requirements installed"

echo ""
echo "=== [7/7] Force re-pin PyTorch 1.13.1 (safety net) ==="
$PIP install torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
  --extra-index-url https://download.pytorch.org/whl/cu117 \
  --force-reinstall --no-deps -q
echo "✅ PyTorch 1.13.1 confirmed"

echo ""
echo "=== Final check ==="
/kaggle/working/py310env/bin/python -c "import torch; print(f'PyTorch: {torch.__version__}')"
/kaggle/working/py310env/bin/python -c "import mmcv; print(f'MMCV:    {mmcv.__version__}')"
echo ""
echo "✅ ALL DEPENDENCIES INSTALLED"


In [ ]:
!/kaggle/working/py310env/bin/pip install unidecode importlib-metadata "langdetect<2.0.0,>=1.0.9" "pyclipper<2.0.0,>=1.2.0" "pypdfium2<5.0.0,>=4.0.0" "weasyprint>=55.0" -q


In [ ]:
!/kaggle/working/py310env/bin/pip install unidecode -q


### 1.4 Verify Installation

In [ ]:
%%bash
/kaggle/working/py310env/bin/python << 'EOF'
import sys
print(f'Python:      {sys.version}')

import torch
print(f'PyTorch:     {torch.__version__}')
print(f'CUDA avail:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:         {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

import torchvision
print(f'TorchVision: {torchvision.__version__}')

import mmcv
print(f'MMCV:        {mmcv.__version__}')

import mmdet
print(f'MMDet:       {mmdet.__version__}')

import cv2
print(f'OpenCV:      {cv2.__version__}')

from Levenshtein import distance
print(f'Levenshtein: OK')

from colorama import Fore
print(f'Colorama:    OK')

import gdown
print(f'gdown:       {gdown.__version__}')

import rasterio
print(f'Rasterio:    {rasterio.__version__}')

import doctr
print(f'doctr:       {doctr.__version__}')

import pytesseract
print(f'Tesseract:   OK')

assert torch.cuda.is_available(), 'No GPU! Go to Settings > Accelerator > GPU T4 x1'
print()
print('🎯 ALL IMPORTS SUCCESSFUL — ENVIRONMENT READY!')
EOF

In [ ]:
!/kaggle/working/py310env/bin/pip install torch==1.13.1+cu117 torchvision==0.14.1+cu117 --extra-index-url https://download.pytorch.org/whl/cu117 --force-reinstall --no-deps -q


In [ ]:
%%bash
PIP=/kaggle/working/py310env/bin/pip

echo "=== Removing broken lightning packages ==="
$PIP uninstall -y pytorch-lightning lightning-fabric lightning-utilities torchmetrics 2>/dev/null

echo ""
echo "=== Installing exact version-matched lightning stack ==="
$PIP install pytorch_lightning==1.9.5 lightning-fabric==1.9.5 "lightning-utilities>=0.6.0,<0.9.0" "torchmetrics>=0.7.0,<1.0.0" --no-deps -q
$PIP install "lightning-utilities>=0.6.0,<0.9.0" "torchmetrics>=0.7.0,<1.0.0" -q

echo ""
echo "=== Force re-pin PyTorch 1.13.1 ==="
$PIP install torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
  --extra-index-url https://download.pytorch.org/whl/cu117 \
  --force-reinstall --no-deps -q

echo ""
echo "=== Verify ==="
/kaggle/working/py310env/bin/python -c "
import torch; print(f'PyTorch:     {torch.__version__}')
import pytorch_lightning as pl; print(f'Lightning:   {pl.__version__}')
print('✅ All good!')
"


In [ ]:
!/kaggle/working/py310env/bin/pip install matplotlib-inline -q
print("✅ Done")

In [ ]:
import os
from pathlib import Path

# ── Fix GPU/CPU mismatch ─────────────────────────────────────────
filepath = '/kaggle/working/doc-matcher/src/unwarp_geotr/inference.py'
with open(filepath, 'r') as f:
    content = f.read()

old = "        out_bm = model(**model_kwargs).detach().cpu()"
new = """        model = model.cuda()
        model_kwargs = {k: v.cuda() if hasattr(v, 'cuda') else v for k, v in model_kwargs.items()}
        out_bm = model(**model_kwargs).detach().cpu()"""

if old in content and 'model = model.cuda()' not in content:
    content = content.replace(old, new)
    with open(filepath, 'w') as f:
        f.write(content)
    print("✅ GPU patch applied")
elif 'model = model.cuda()' in content:
    print("✅ GPU patch already applied")
else:
    print("⚠️  Pattern not found — patch failed")

# ── Set environment ──────────────────────────────────────────────
TORCH_LIB = "/kaggle/working/py310env/lib/python3.10/site-packages/torch/lib"
os.environ["LD_LIBRARY_PATH"] = f"{TORCH_LIB}:{os.environ.get('LD_LIBRARY_PATH', '')}"
os.environ["MPLBACKEND"] = "Agg"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("✅ Environment variables set")

# ── Run inference ────────────────────────────────────────────────
%cd /kaggle/working/doc-matcher
!/kaggle/working/py310env/bin/python inference.py --model docmatcher@inv3d --dataset inv3d_real --gpu 0

---
## 2. Inference — Dewarping Document Images

We run inference on the **Inv3DReal** dataset (360 real-world warped document images) using multiple models:

| Model | Type | Description |
|-------|------|-------------|
| `identity` | Baseline | No dewarping (original warped image) |
| `dewarpnet@inv3d` | Learning-based | DewarpNet trained on Inv3D |
| `geotr@inv3d` | Learning-based | GeoTr trained on Inv3D |
| `geotr_template@inv3d` | Template-based | GeoTr with template guidance |
| `docmatcher@inv3d` | **Proposed** | DocMatcher (full pipeline) |

Models and dataset are **downloaded automatically**.

> 💡 All inference commands use `/kaggle/working/py310env/bin/python` to run in the Python 3.10 environment.

### 2.1 Run DocMatcher (Proposed Method)
Full pipeline: SAM → LineFormer → Homography → GeoTr → LineFormer → LightGlue → Correspondence Unwarp

### 2.2 Run Baseline Models

In [ ]:
%%time
%cd /kaggle/working/doc-matcher
print("="*60)
print("Running: identity (no dewarping baseline)")
print("="*60)
!/kaggle/working/py310env/bin/python inference.py --model identity --dataset inv3d_real --gpu 0

In [ ]:
%%time
%cd /kaggle/working/doc-matcher
print("="*60)
print("Running: dewarpnet@inv3d")
print("="*60)
!/kaggle/working/py310env/bin/python inference.py --model dewarpnet@inv3d --dataset inv3d_real --gpu 0

In [ ]:
%%time
%cd /kaggle/working/doc-matcher
print("="*60)
print("Running: geotr@inv3d")
print("="*60)
!/kaggle/working/py310env/bin/python inference.py --model geotr@inv3d --dataset inv3d_real --gpu 0

In [ ]:
%%time
%cd /kaggle/working/doc-matcher
print("="*60)
print("Running: geotr_template@inv3d")
print("="*60)
!/kaggle/working/py310env/bin/python inference.py --model geotr_template@inv3d --dataset inv3d_real --gpu 0

---
## 3. Qualitative Results — Visual Comparison

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import numpy as np
import random

%cd /kaggle/working/doc-matcher

models = {
    'Identity\n(No Dewarping)': 'output/inv3d_real-identity',
    'DewarpNet': 'output/inv3d_real-dewarpnet@inv3d',
    'GeoTr': 'output/inv3d_real-geotr@inv3d',
    'GeoTr\n+Template': 'output/inv3d_real-geotr_template@inv3d',
    'DocMatcher\n(Proposed)': 'output/inv3d_real-docmatcher@inv3d',
}

available_models = {k: v for k, v in models.items() if Path(v).exists()}

if not available_models:
    print("No output directories found. Run inference cells first.")
else:
    first_model_dir = Path(list(available_models.values())[0])
    sample_dirs = sorted([d for d in first_model_dir.iterdir() if d.is_dir()])

    num_display = min(6, len(sample_dirs))
    random.seed(42)
    selected_samples = random.sample(sample_dirs, num_display)

    num_models = len(available_models)
    fig, axes = plt.subplots(num_display, num_models, figsize=(4 * num_models, 4 * num_display))
    if num_display == 1:
        axes = axes.reshape(1, -1)

    for col, (model_name, model_dir) in enumerate(available_models.items()):
        axes[0, col].set_title(model_name, fontsize=14, fontweight='bold',
                               color='green' if 'Proposed' in model_name else 'black')

    for row, sample_path in enumerate(selected_samples):
        sample_name = sample_path.name
        for col, (model_name, model_dir) in enumerate(available_models.items()):
            ax = axes[row, col]
            sample_dir = Path(model_dir) / sample_name
            norm_files = list(sample_dir.glob('norm_image.*')) if sample_dir.exists() else []
            if norm_files:
                ax.imshow(mpimg.imread(str(norm_files[0])))
            else:
                ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes)
            ax.axis('off')

    plt.suptitle('Document Dewarping — Model Comparison', fontsize=18, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('/kaggle/working/doc-matcher/output/qualitative_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\nSaved to output/qualitative_comparison.png")

### 3.1 Detailed Single-Sample View

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

%cd /kaggle/working/doc-matcher

available_models = {k: v for k, v in models.items() if Path(v).exists()}

if available_models:
    first_model_dir = Path(list(available_models.values())[0])
    sample_dirs = sorted([d for d in first_model_dir.iterdir() if d.is_dir()])

    if sample_dirs:
        sample = sample_dirs[0]
        sample_name = sample.name

        fig, axes = plt.subplots(1, len(available_models), figsize=(5 * len(available_models), 5))
        if len(available_models) == 1:
            axes = [axes]

        for idx, (model_name, model_dir) in enumerate(available_models.items()):
            sample_dir = Path(model_dir) / sample_name
            for img_pattern in ['norm_image.*', 'true_image.*']:
                img_files = list(sample_dir.glob(img_pattern)) if sample_dir.exists() else []
                if img_files:
                    axes[idx].imshow(mpimg.imread(str(img_files[0])))
                    break
            axes[idx].set_title(model_name, fontsize=13, fontweight='bold',
                                 color='green' if 'Proposed' in model_name else 'black')
            axes[idx].axis('off')

        plt.suptitle(f'Sample: {sample_name}', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()

---
## 4. Quantitative Evaluation

| Metric | Description | Better |
|--------|-------------|--------|
| **MS-SSIM** | Multi-Scale Structural Similarity | Higher ↑ |
| **LPIPS** | Learned Perceptual Image Patch Similarity | Lower ↓ |
| **CER** | Character Error Rate (OCR quality) | Lower ↓ |
| **ED** | Edit Distance (text accuracy) | Lower ↓ |

In [ ]:
%cd /kaggle/working/doc-matcher
from pathlib import Path

output_dir = Path('output')
runs = sorted([d.name for d in output_dir.iterdir() if d.is_dir()])
print(f"Available runs: {runs}")

for run_name in runs:
    print(f"\n{'='*60}")
    print(f"Evaluating: {run_name}")
    print(f"{'='*60}")
    !/kaggle/working/py310env/bin/python eval.py --run {run_name}

### 4.1 Results Summary Table

In [ ]:
import pandas as pd
from pathlib import Path

%cd /kaggle/working/doc-matcher

output_dir = Path('output')
results = []

model_display_names = {
    'inv3d_real-identity': 'Identity (No Dewarping)',
    'inv3d_real-dewarpnet@inv3d': 'DewarpNet',
    'inv3d_real-geotr@inv3d': 'GeoTr',
    'inv3d_real-geotr_template@inv3d': 'GeoTr + Template',
    'inv3d_real-geotr_template_large@inv3d': 'GeoTr + Template (Large)',
    'inv3d_real-docmatcher@inv3d': 'DocMatcher (Proposed)',
}

for run_dir in sorted(output_dir.iterdir()):
    summary_file = run_dir / 'results_summary.csv'
    if summary_file.exists():
        summary = pd.read_csv(summary_file, header=None, index_col=0)
        row = {'Model': model_display_names.get(run_dir.name, run_dir.name)}
        for metric, value in summary.iterrows():
            row[metric] = value.iloc[0]
        results.append(row)

if results:
    df = pd.DataFrame(results).set_index('Model')
    print("\n" + "="*70)
    print("       QUANTITATIVE RESULTS — Inv3DReal Dataset")
    print("="*70)
    print(df.to_string())
    print("\n📊 Higher is better: MS-SSIM")
    print("📊 Lower is better: LPIPS, CER, ED")
    df.to_csv('output/comparison_results.csv')
    print("\n✅ Saved to output/comparison_results.csv")
else:
    print("No evaluation results yet. Run the evaluation cell first.")

### 4.2 Results Bar Charts

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import numpy as np

%cd /kaggle/working/doc-matcher

results_file = Path('output/comparison_results.csv')
if results_file.exists():
    df = pd.read_csv(results_file, index_col='Model')
    metrics = df.columns.tolist()
    higher_is_better = {'ms_ssim'}
    colors = ['#95a5a6', '#e74c3c', '#3498db', '#9b59b6', '#2ecc71']

    fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 6))
    if len(metrics) == 1:
        axes = [axes]

    for idx, metric in enumerate(metrics):
        ax = axes[idx]
        values = df[metric].values
        model_names = list(df.index)
        best_idx = np.argmax(values) if metric in higher_is_better else np.argmin(values)
        arrow = 'Higher' if metric in higher_is_better else 'Lower'
        bar_colors = [colors[i % len(colors)] for i in range(len(values))]
        bar_colors[best_idx] = '#27ae60'

        bars = ax.barh(model_names, values, color=bar_colors, edgecolor='white', linewidth=0.5)
        for bar, val in zip(bars, values):
            ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                    f'{val:.4f}', va='center', fontsize=10)
        ax.set_title(f'{metric.upper()} ({arrow} is better)', fontsize=14, fontweight='bold')
        ax.set_xlabel('Score', fontsize=12)
        ax.invert_yaxis()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.suptitle('Quantitative Comparison — Inv3DReal Dataset', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('/kaggle/working/doc-matcher/output/quantitative_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\nSaved to output/quantitative_comparison.png")
else:
    print("No comparison results yet. Run evaluation cells first.")

---
## 5. Quick Demo — Example Dataset

The repository includes a small example dataset (2 images) for quick testing.

In [ ]:
filepath = '/kaggle/working/doc-matcher/src/unwarp_geotr/models/geotr/geotr_template_large.py'
with open(filepath, 'r') as f:
    content = f.read()

# Patch 1 — backbone forward (line 89)
old1 = '    def forward(self, image):\n        check_tensor(image, "n 3 600 600")'
new1 = '    def forward(self, image):\n        self.backbone = self.backbone.to(image.device)\n        self.reduce_conv = self.reduce_conv.to(image.device)\n        check_tensor(image, "n 3 600 600")'

# Patch 2 — main model forward (line 135)
old2 = '    def forward(self, image1, template):\n        fmap1 = self.fnet1(image1)'
new2 = '    def forward(self, image1, template):\n        self.to(image1.device)\n        fmap1 = self.fnet1(image1)'

patched = False
if old1 in content and 'self.backbone = self.backbone.to' not in content:
    content = content.replace(old1, new1)
    print("✅ Patch 1 applied — backbone forward")
    patched = True
else:
    print("⚠️ Patch 1 skipped — already applied or not found")

if old2 in content and 'self.to(image1.device)' not in content:
    content = content.replace(old2, new2)
    print("✅ Patch 2 applied — main model forward")
    patched = True
else:
    print("⚠️ Patch 2 skipped — already applied or not found")

if patched:
    with open(filepath, 'w') as f:
        f.write(content)
    print("✅ File saved")

# Verify
with open(filepath, 'r') as f:
    lines = f.readlines()
for i, line in enumerate(lines[88:95], start=89):
    print(f"{i}: {repr(line)}")
for i, line in enumerate(lines[134:139], start=135):
    print(f"{i}: {repr(line)}")

In [ ]:
import os

TORCH_LIB = "/kaggle/working/py310env/lib/python3.10/site-packages/torch/lib"
os.environ["LD_LIBRARY_PATH"] = f"{TORCH_LIB}:{os.environ.get('LD_LIBRARY_PATH', '')}"
os.environ["MPLBACKEND"] = "Agg"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

%cd /kaggle/working/doc-matcher
!/kaggle/working/py310env/bin/python inference.py --model docmatcher@inv3d --dataset example --gpu 0

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

%cd /kaggle/working/doc-matcher

example_output = Path('output/example-docmatcher@inv3d')
example_input = Path('input/custom_datasets/example')

if example_output.exists():
    sample_dirs = sorted([d for d in example_output.iterdir() if d.is_dir()])
    for sample_dir in sample_dirs:
        sample_name = sample_dir.name
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        warped_files = sorted(example_input.glob('image_*'))
        template_files = sorted(example_input.glob('template_*'))
        norm_files = list(sample_dir.glob('norm_image.*'))

        idx = int(sample_name.split('_')[-1]) - 1 if sample_name.split('_')[-1].isdigit() else 0
        if idx < len(warped_files):
            axes[0].imshow(mpimg.imread(str(warped_files[idx])))
        axes[0].set_title('Warped Input', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        if idx < len(template_files):
            axes[1].imshow(mpimg.imread(str(template_files[idx])))
        axes[1].set_title('Template', fontsize=14, fontweight='bold')
        axes[1].axis('off')

        if norm_files:
            axes[2].imshow(mpimg.imread(str(norm_files[0])))
        axes[2].set_title('DocMatcher Output', fontsize=14, fontweight='bold', color='green')
        axes[2].axis('off')

        plt.suptitle(f'DocMatcher Dewarping — {sample_name}', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
else:
    print("Run the example inference cell first.")

---
## 6. Download / Save Results

> 💡 On Kaggle, all files saved to `/kaggle/working/` are automatically available in the **Output** tab on the right panel. You can download them directly from there.
>
> The cell below also creates a zip archive at `/kaggle/working/docmatcher_results.zip` for convenience.

In [ ]:
import shutil
from pathlib import Path

%cd /kaggle/working/doc-matcher

output_zip = '/kaggle/working/docmatcher_results'
shutil.make_archive(output_zip, 'zip', 'output')
print(f"Results archived to {output_zip}.zip")
print(f"Size: {Path(output_zip + '.zip').stat().st_size / (1024*1024):.1f} MB")
print()
print("✅ Download docmatcher_results.zip from the Kaggle Output tab (right panel).")

In [ ]:
from IPython.display import FileLink
FileLink('/kaggle/working/docmatcher_results.zip')

---
## 7. Citation

```bibtex
@InProceedings{hertlein2025docmatcher,
    author    = {Hertlein, Felix and Naumann, Alexander and Sure-Vetter, York},
    title     = {DocMatcher: Document Image Dewarping via Structural and Textual Line Matching},
    booktitle = {Proceedings of the Winter Conference on Applications of Computer Vision (WACV)},
    month     = {February},
    year      = {2025},
    pages     = {5771-5780}
}
```

---
*This notebook reproduces results from the DocMatcher paper (WACV 2025). All models and data are downloaded automatically.*